**Hubbard (DFT+U+V) mode is currently unsupported with aiida-quantumespresso 5.x** (aiida-hubbard has no compatible release yet, see aiida-hubbard PR #119). This notebook targets plugin **v0.6** (git tag) with aiida-quantumespresso 4.17 and aiida-hubbard 0.5.

# OCVWorkChain: extended Hubbard (DFT+U+V)

Computes voltages with on-site U and inter-site V Hubbard corrections. U and V are converged
self-consistently with `hp.x` (via `aiida-hubbard`) on the discharged and charged unitcells, then
transferred to the supercells for the fixed-U/V relaxations. Requirements beyond the vanilla
workflow: an `hp.x` code and the `aiida-hubbard` plugin (>= 0.5).

The example structures used below are bundled with the repository as an AiiDA archive.
Import them once with

```
verdi archive import test/structures.aiida
```

or load your own structure instead, e.g. `orm.StructureData(ase=ase.io.read('my_cathode.cif'))`.

## Loading libraries

In [ ]:
from aiida import load_profile, orm
## Indicate your profile name here
your_profile_name = 'develop'
load_profile(your_profile_name)
from aiida.plugins import WorkflowFactory
from aiida.engine import submit

## Codes, structure and Hubbard specification

In [ ]:
## Codes: pw.x AND hp.x are required for Hubbard mode
pw_code = orm.load_code(label='pw@your_computer')
hp_code = orm.load_code(label='hp@your_computer')

## default resources
time, num_machines, num_mpiprocs_per_machine, num_cores_per_mpiproc, npool = 83200, 2, 128, 1, 8
hp_time, hp_num_machines, hp_num_mpiprocs_per_machine = 43200, 1, 32

## Load a bundled test structure (import test/structures.aiida first) or your own
structure = orm.load_node('3dd4a60a-a5d5-48b6-a8b1-3e082664622a') # LiFePO4
# structure = orm.load_node('096d9d96-7f66-442b-a27f-2660572808ea') # LiCoO2

## Hubbard spec: onsite U on the redox-active metal and intersite V.
## Trailing values are initial seeds in eV are strongly recommended.
## Literature values computed with the same atomic projector convention, e.g. from DFPT studies.
HUBBARD_SPEC = {'U': [['Fe', '3d', 5.3]],               # LiFePO4: Timrov et al., PRX Energy 1, 033003
                'V': [['Fe', '3d', 'O', '2p', 0.7]]}
# HUBBARD_SPEC = {'U': [['Co', '3d', 6.91]],            # Floris et al., PRB 101, 064305 (2020)
#                 'V': [['Co', '3d', 'O', '2p']]}        

## Launching the Hubbard OCVWorkChain

In [ ]:
## Submitting the Hubbard OCVWorkChain
## Passing hp_code together with the `hubbard` spec switches the workchain into DFT+U+V mode.
## Without a bulk_cation_structure the stored DFT_energy_bulk_<cation> from the protocol is used.
OCVWorkChain = WorkflowFactory('quantumespresso.ocv.ocvwc')

overrides = {
    'ocv_parameters': {
        'cation': 'Li',
        'hubbard': HUBBARD_SPEC,
        'do_low_SOC_OCV': False,    # average OCV only; set True to also compute the SOC voltages
        'do_high_SOC_OCV': False,
    },
    ## local-TF mixing measurably improves transition-metal-oxide SCF convergence in the SC loop
    'hubbard_sc': {
        'scf':   {'pw': {'parameters': {'ELECTRONS': {'mixing_mode': 'local-TF', 'mixing_beta': 0.1}}}},
        'relax': {'base': {'pw': {'parameters': {'ELECTRONS': {'mixing_mode': 'local-TF', 'mixing_beta': 0.1}}}}},
    },
}

builder = OCVWorkChain.get_builder_from_protocol(code=pw_code, hp_code=hp_code, structure=structure, 
                                                 protocol='balanced', overrides=overrides)
builder.clean_workdir = orm.Bool(False)   # keep restart folders of the SC loop

## self-consistent DFT+U+V loop resources
for pw in (builder.hubbard_sc.scf.pw, builder.hubbard_sc.relax.base.pw):
    pw.metadata['options']['max_wallclock_seconds'] = time
    pw.metadata['options']['resources']['num_machines'] = num_machines
    pw.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
    pw.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
    pw.parallelization = orm.Dict(dict={'npool': npool})

hp = builder.hubbard_sc.hubbard.hp
hp.metadata['options']['max_wallclock_seconds'] = hp_time
hp.metadata['options']['resources']['num_machines'] = hp_num_machines
hp.metadata['options']['resources']['num_mpiprocs_per_machine'] = hp_num_mpiprocs_per_machine
hp.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc

## fixed-U/V relaxations (discharged + charged unitcells and, if enabled, the SOC supercells)
for pw in (builder.ocv_relax.base.pw, builder.ocv_relax.base_final_scf.pw):
    pw.metadata['options']['max_wallclock_seconds'] = time
    pw.metadata['options']['resources']['num_machines'] = num_machines
    pw.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
    pw.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
    pw.parallelization = orm.Dict(dict={'npool': npool})

node = submit(builder)
print(f'Submitted Hubbard OCVWorkChain PK={node.pk}')

## Advanced options

In [ ]:
## Optional settings

## 1. Collinear magnetism: the spin settings apply to the whole workchain including the SC loop.
##    The charged branches strip the cation's moment automatically.
from aiida_quantumespresso.common.types import SpinType
builder = OCVWorkChain.get_builder_from_protocol(
    code=pw_code, hp_code=hp_code, structure=structure, protocol='balanced', overrides=overrides,
    spin_type=SpinType.COLLINEAR,
    initial_magnetic_moments={'Fe': 1.0, 'Li': 0.0, 'O': 0.0})   # one entry per kind, in Bohr magnetons

## 2. Per-branch total-magnetization constraints (µB): pin the `pw.x` total moment of one branch's
##    calculations (SC-loop relax + fixed-U/V relax/scf; never the SC smearing scf). 
##    The two endpoints generally need different values. 
#     Required with nspin=2 and fixed occupations.
overrides['ocv_parameters']['tot_magnetization_discharged'] = 16.0
overrides['ocv_parameters']['tot_magnetization_charged'] = 20.0

## 3. Changing occupations requires the per-branch constraints above to be set.
for pw in (builder.ocv_relax.base.pw, builder.ocv_relax.base_final_scf.pw):
    pw.parameters['SYSTEM']['occupations'] = 'fixed'

## 4. Restarts: skip a converged self-consistent loop by passing its result back in.
builder.discharged_hubbard_structure = orm.load_node(PK).outputs.discharged_hubbard_structure
builder.charged_hubbard_structure = orm.load_node(PK).outputs.charged_hubbard_structure

## Results

In [ ]:
node = orm.load_node(PK)
print(node.outputs.open_circuit_voltages.get_dict())
node.outputs.discharged_hubbard_structure   # converged U/V, reusable for restarts
node.outputs.charged_hubbard_structure